simple parallel workflow 

In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv


In [2]:
load_dotenv()



True

In [3]:
class batsmanState(TypedDict):
    runs : int 
    balls : int
    fours : int 
    sixes : int

    sr : float
    bpb : float
    boundary_percentage : float
    summary : str

In [4]:
def generate_sr(state : batsmanState) -> batsmanState:
    sr = (state['runs']/state['balls'])*100

    return {'sr' : sr}

In [5]:
def generate_bpb(state : batsmanState) -> batsmanState:
    bpb = state['balls']/(state['fours']+state['sixes'])

    return {'bpb': bpb}

In [6]:
def generate_boudary_p(state : batsmanState) -> batsmanState:
    boundary_percent = (((state['fours']*4) + (state['sixes']*6))/state['runs'])*100

    return {'boundary_percentage': boundary_percent}

In [23]:
def summary(state : batsmanState) -> batsmanState:
    summary = f""" 
        \t\tMATCH STATISTICS (INSIGHTS)\t\t\n\n
        strike-rate - {state['sr']} \n
        balls-per-boundary - {state['bpb']} \n 
        boundary-percentage - {state['boundary_percentage']}
    """

    return {'summary': summary}

In [24]:
graph = StateGraph(batsmanState)

#add node 
graph.add_node('generate_sr',generate_sr)
graph.add_node('generate_bpb',generate_bpb)
graph.add_node('generate_boudary_p',generate_boudary_p)
graph.add_node('summary', summary)

# add edges
graph.add_edge(START , 'generate_sr')
graph.add_edge(START , 'generate_bpb')
graph.add_edge(START , 'generate_boudary_p')

graph.add_edge('generate_sr', 'summary')
graph.add_edge('generate_bpb', 'summary')
graph.add_edge('generate_boudary_p', 'summary')

graph.add_edge('summary',END)



# see the workflow and compile as well 
# graph.compile()
workflow = graph.compile()


In [25]:
initial_state = {'runs' : 100 , 'balls': 50 , 'fours': 6 , 'sixes' : 4}

final_state = workflow.invoke(initial_state)

In [27]:
final_state

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percentage': 48.0,
 'summary': ' \n        \t\tMATCH STATISTICS (INSIGHTS)\t\t\n\n\n        strike-rate - 200.0 \n\n        balls-per-boundary - 5.0 \n \n        boundary-percentage - 48.0\n    '}

In [26]:
print(final_state['summary'])

 
        		MATCH STATISTICS (INSIGHTS)		


        strike-rate - 200.0 

        balls-per-boundary - 5.0 
 
        boundary-percentage - 48.0
    
